# Component Ablation (long model) — NCKH Table 3 (GH-71)

Train lại long model v2.2 với ĐÚNG config gốc, chỉ đổi 1 thành phần mỗi run.
So với baseline full model **MAE 1.52 / RMSE 1.97** (checkpoint v2.2).

**CÁCH CHẠY — 3 lần, mỗi lần 1 variant (~3–5h/run, vừa 1 session):**
1. Sửa `VARIANT` ở cell đầu = `"pooling_last"` → Save Version → Save & Run All
2. Đổi `VARIANT = "d_state16"` → Save Version → Save & Run All
3. Đổi `VARIANT = "no_weighted_loss"` → Save Version → Save & Run All

Mỗi run xuất `ablation_<variant>.zip` trong tab Output → tải về
`logs/nckh/ablation/`.

**Setup:** Accelerator = **GPU T4 x2** (KHÔNG P100) · Internet On ·
Add Data `nasa-battery-dataset` · Secret `GITHUB_TOKEN`.

⚠️ KHÔNG commit weights variant vào repo — chỉ lấy số MAE/RMSE.

In [ ]:
# ===== CHON VARIANT CHO RUN NAY =====
VARIANT = "pooling_last"   # "pooling_last" | "d_state16" | "no_weighted_loss"

assert VARIANT in {"pooling_last", "d_state16", "no_weighted_loss"}
print("VARIANT =", VARIANT)

## 1. GPU + clone repo (branch `dev`)

In [ ]:
import subprocess

import torch
assert torch.cuda.is_available(), "Settings -> Accelerator -> GPU T4 x2!"

REPO_URL = "github.com/GSU26SE55/ai-module.git"
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    clone_url = f"https://{token}@{REPO_URL}"
except Exception:
    clone_url = f"https://{REPO_URL}"

subprocess.run(["git", "clone", "--branch", "dev", "--single-branch",
                clone_url, "/kaggle/working/ai-module"], check=True)
subprocess.run(["git", "-C", "/kaggle/working/ai-module", "remote", "set-url",
                "origin", f"https://{REPO_URL}"], check=True)
commit = subprocess.run(["git", "-C", "/kaggle/working/ai-module", "rev-parse", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print("Commit:", commit)

## 2. Deps + ép split CŨ 23/2/1 + vá variant

- Split: v2.2 train TRƯỚC GH-88 → phải trả B0047 về VAL (giống protocol bài báo),
  nếu không Δ MAE lẫn lộn giữa "đổi thành phần" và "đổi data".
- Variant patch: pooling hardcode trong `train.py` (flag `--pooling` chỉ cho RUL);
  d_state là hằng `LONG_D_STATE` trong `config.py`.

In [ ]:
%pip install -q scipy scikit-learn

import os

REPO = "/kaggle/working/ai-module"
DATASET = "/kaggle/input/nasa-battery-dataset/cleaned_dataset"
PROCESSED = "/kaggle/working/processed_long"

if not os.path.isfile(f"{DATASET}/metadata.csv"):
    import glob
    hits = glob.glob("/kaggle/input/**/metadata.csv", recursive=True)
    assert hits, "Khong tim thay metadata.csv"
    DATASET = os.path.dirname(hits[0])
os.chdir(REPO)

# --- ep split CU: tra B0047 ve VAL ---
prep = f"{REPO}/scripts/preprocess.py"
src = open(prep, encoding="utf-8").read()
b0047_line = '    "B0047",  # 4\u00b0C, SOH 0-83.7% \u2014 GH-88: fills the missing 4\u00b0C high-SOH region\n'
assert b0047_line in src and 'VAL_IDS = ["B0046"]' in src, "preprocess.py da doi — cap nhat patch"
src = src.replace(b0047_line, "")
src = src.replace('VAL_IDS = ["B0046"]', 'VAL_IDS = ["B0046", "B0047"]')
open(prep, "w", encoding="utf-8").write(src)
print("Split cu OK: 23 train / 2 val / 1 test")

# --- va variant ---
if VARIANT == "pooling_last":
    tr = f"{REPO}/scripts/train.py"
    s = open(tr, encoding="utf-8").read()
    old = '        pooling="attention",'
    assert s.count(old) == 1, f"pooling hardcode thay doi ({s.count(old)} match) — kiem tra train.py"
    open(tr, "w", encoding="utf-8").write(s.replace(old, '        pooling="last",'))
    print("Patched: long model pooling attention -> last")
elif VARIANT == "d_state16":
    cf = f"{REPO}/src/core/config.py"
    s = open(cf, encoding="utf-8").read()
    assert "LONG_D_STATE = 32" in s
    open(cf, "w", encoding="utf-8").write(s.replace("LONG_D_STATE = 32", "LONG_D_STATE = 16"))
    print("Patched: LONG_D_STATE 32 -> 16")
else:
    print("no_weighted_loss: khong va code — chi bo flag --weighted-loss o cell train")

## 3. Preprocess long (L=4096, split cũ)

In [ ]:
!python scripts/preprocess_long.py --data-dir "{DATASET}" --output-dir "{PROCESSED}"

## 4. Train — đúng lệnh tái tạo v2.2, chỉ khác 1 knob

Flags khớp metadata checkpoint v2.2: patch-stride 8, cosine-t0 80,
weight-decay 3e-4, dropout 0.3, jitter 0.0075, weighted-loss
(bỏ nếu VARIANT=no_weighted_loss). Các default còn lại giữ nguyên.

In [ ]:
wl_flag = "" if VARIANT == "no_weighted_loss" else "--weighted-loss"

!python scripts/train.py --long \
    --data-dir "{PROCESSED}" \
    --patch-stride 8 --cosine-t0 80 --weight-decay 3e-4 \
    --dropout 0.3 --jitter 0.0075 {wl_flag} \
    --compile --num-workers 4 \
    --log-dir /kaggle/working/logs

## 5. Lấy test MAE/RMSE từ checkpoint + lưu kết quả

In [ ]:
import glob
import json
import shutil

import torch

ckpts = sorted(glob.glob(f"{REPO}/models/weights/soh_mamba_long_v*.pth"), key=os.path.getmtime)
ckpt_path = ckpts[-1]
meta = {k: v for k, v in torch.load(ckpt_path, map_location="cpu", weights_only=False).items()
        if k != "model_state_dict"}
print(json.dumps({k: str(v) for k, v in meta.items()}, indent=2))

out_dir = f"/kaggle/working/ablation_{VARIANT}"
os.makedirs(out_dir, exist_ok=True)
result = {
    "variant": VARIANT,
    "commit": commit,
    "protocol": "OLD split 23/2/1 (B0047 in VAL), seed 42, L=4096",
    "baseline_full_v22": {"test_mae_pct": 1.5232, "test_rmse_pct": 1.9708},
    "test_mae_pct": float(meta.get("test_mae", float("nan"))),
    "test_rmse_pct": float(meta.get("test_rmse", float("nan"))),
    "checkpoint_meta": {k: str(v) for k, v in meta.items()},
}
with open(f"{out_dir}/result.json", "w") as f:
    json.dump(result, f, indent=2)
for lg in glob.glob("/kaggle/working/logs/*.log"):
    shutil.copy(lg, out_dir)
shutil.make_archive(f"/kaggle/working/ablation_{VARIANT}", "zip", out_dir)
print(f"\n=== {VARIANT}: MAE {result['test_mae_pct']:.4f}% | RMSE {result['test_rmse_pct']:.4f}% "
      f"(baseline 1.5232/1.9708) ===")
print(f"Tai ablation_{VARIANT}.zip tu tab Output -> logs/nckh/ablation/")